# Dryad Exploratory Data Analysis

This notebook performs a careful, robust exploratory analysis of the Dryad measurement files (`pipeline_input.mat` or `pipeline_input.npz`, plus `fbg_group_delay.npy` if present). It produces numeric summaries, diagnostic plots, and a short automated checklist that answers: **Are these intensity traces suitable for a time‑domain Gerchberg–Saxton (TD‑GS) reconstruction?**

Run the cells in order. The notebook is defensive: it coerces dtypes, handles missing files, and writes a small `figures/` folder and `figures/explore_summary.json` with numeric diagnostics.

In [5]:
!pip install seaborn
!pip install tqdm


[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\mrjel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.1.2
[notice] To update, run: C:\Users\mrjel\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [6]:
# Install minimal dependencies (uncomment if running in a fresh environment)
# !pip install -q numpy scipy matplotlib seaborn tqdm librosa
import json
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import io as spio
from scipy.signal import welch, spectrogram, correlate
from tqdm import tqdm
sns.set(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

DATA_DIR = Path('data/raw')
FALLBACK_DIR = Path('C:/Temp/dryad_output')
OUT_DIR = Path('figures')
OUT_DIR.mkdir(parents=True, exist_ok=True)

print('Notebook ready. Data dir:', DATA_DIR)


Notebook ready. Data dir: data\raw


## 1) Locate and load pipeline input
This cell looks for `pipeline_input.mat` or `pipeline_input.npz` in `data/raw` (then falls back to `C:/Temp/dryad_output`). It loads `i1`, `i2`, `time`, and `group_delay` if present.

In [7]:
def find_pipeline_file():
    candidates = [
        DATA_DIR / 'pipeline_input.npz',
        DATA_DIR / 'pipeline_input.mat',
        FALLBACK_DIR / 'pipeline_input.npz',
        FALLBACK_DIR / 'pipeline_input.mat'
    ]
    for p in candidates:
        if p.exists():
            return p
    return None

def load_pipeline(path: Path):
    out = {}
    if path.suffix.lower() == '.npz':
        d = np.load(str(path), allow_pickle=True)
        for k in d.files:
            out[k] = np.asarray(d[k])
    elif path.suffix.lower() == '.mat':
        if spio is None:
            raise RuntimeError('scipy not available to read .mat files')
        m = spio.loadmat(str(path))
        for k in ('i1','i2','time','group_delay'):
            if k in m:
                out[k] = np.asarray(m[k]).squeeze()
    else:
        raise RuntimeError('Unsupported file type: ' + str(path))
    # also try to load fbg_group_delay.npy if present
    fbg = path.parent / 'fbg_group_delay.npy'
    if fbg.exists():
        out['fbg_group_delay'] = np.load(str(fbg))
    # measurement metadata
    meta = path.parent / 'measurement_metadata.json'
    if meta.exists():
        try:
            out['_metadata'] = json.loads(meta.read_text())
        except Exception:
            out['_metadata'] = None
    return out

p = find_pipeline_file()
if p is None:
    raise FileNotFoundError('No pipeline_input.mat or pipeline_input.npz found in data/raw or C:/Temp/dryad_output')
print('Loading:', p)
data = load_pipeline(p)
print('Loaded keys:', list(data.keys()))


FileNotFoundError: No pipeline_input.mat or pipeline_input.npz found in data/raw or C:/Temp/dryad_output

## 2) Coerce arrays to numeric 1‑D and print basic stats
This cell forces numeric dtypes (avoids object arrays), computes min/max/mean/std, and prints shapes and dtypes.

In [ ]:
def to_1d_numeric(x, name):
    if x is None:
        return None
    a = np.asarray(x)
    if a.size == 0:
        return None
    a = a.ravel()
    try:
        return a.astype(np.float64)
    except Exception:
        # try elementwise conversion
        return np.array([float(v) for v in a], dtype=np.float64)

i1 = to_1d_numeric(data.get('i1'), 'i1')
i2 = to_1d_numeric(data.get('i2'), 'i2')
time_arr = to_1d_numeric(data.get('time'), 'time')
group_delay = to_1d_numeric(data.get('group_delay'), 'group_delay')
fbg_group_delay = to_1d_numeric(data.get('fbg_group_delay'), 'fbg_group_delay')

def stats(arr):
    return {
        'shape': None if arr is None else arr.shape,
        'dtype': None if arr is None else str(arr.dtype),
        'min': None if arr is None else float(np.nanmin(arr)),
        'max': None if arr is None else float(np.nanmax(arr)),
        'mean': None if arr is None else float(np.nanmean(arr)),
        'std': None if arr is None else float(np.nanstd(arr))
    }

summary = {
    'i1': stats(i1),
    'i2': stats(i2),
    'time': stats(time_arr),
    'group_delay': stats(group_delay),
    'fbg_group_delay': stats(fbg_group_delay),
    '_metadata': data.get('_metadata', None)
}
print(json.dumps(summary, indent=2))


## 3) Quick automated checks for TD‑GS suitability
We apply a short checklist. Each check returns PASS / WARN / FAIL with a short explanation.
Criteria used (practical):
- **Presence**: `i1` must exist and be 1‑D numeric.
- **Length**: signal length should be >= 512 samples for stable FFT; longer is better.
- **No NaNs / finite**: data must be finite.
- **SNR proxy**: signal RMS / noise RMS > 10 recommended (heuristic).
- **Diversity**: if `i2` exists and is not identical to `i1`, it can serve as a second measurement (good for Solli et al. two‑measurement GS).
- **Group delay**: if present, length should be compatible with FFT bins or be resampleable.


In [ ]:
def check_presence(arr, name):
    if arr is None:
        return ('FAIL', f'{name} missing')
    return ('PASS', f'{name} present, length={arr.size}')

def check_length(arr, min_len=512):
    if arr is None:
        return ('FAIL', 'missing')
    if arr.size < min_len:
        return ('WARN', f'length {arr.size} < {min_len} (may be low resolution)')
    return ('PASS', f'length {arr.size}')

def check_finite(arr, name):
    if arr is None:
        return ('FAIL', f'{name} missing')
    if not np.isfinite(arr).all():
        return ('FAIL', f'{name} contains non-finite values')
    return ('PASS', f'{name} all finite')

def snr_proxy(x):
    x = np.asarray(x)
    med = np.median(x)
    noise = x - med
    noise_rms = np.sqrt(np.mean((noise - np.median(noise))**2))
    sig_rms = np.sqrt(np.mean((x)**2))
    return float(sig_rms / (noise_rms + 1e-20))

checks = {}
checks['i1_presence'] = check_presence(i1, 'i1')
checks['i1_length'] = check_length(i1, min_len=512)
checks['i1_finite'] = check_finite(i1, 'i1')
checks['i1_snr_proxy'] = ('PASS' if i1 is not None and snr_proxy(i1) > 10 else 'WARN', f'snr_proxy={snr_proxy(i1) if i1 is not None else None}')

if i2 is not None:
    # compare i1 and i2 similarity
    try:
        corr = np.corrcoef(i1, i2)[0,1]
    except Exception:
        corr = None
    checks['i2_presence'] = check_presence(i2, 'i2')
    checks['i2_similarity'] = ('WARN' if corr is not None and corr > 0.98 else 'PASS', f'corr={corr:.3f}' if corr is not None else 'corr_unknown')
else:
    checks['i2_presence'] = ('FAIL', 'i2 missing')
    checks['i2_similarity'] = ('N/A', 'no i2')

if group_delay is not None:
    checks['group_delay_len'] = ('PASS' if group_delay.size > 10 else 'WARN', f'len={group_delay.size}')
else:
    checks['group_delay_len'] = ('N/A', 'no group_delay')

print(json.dumps(checks, indent=2))


## 4) Visual diagnostics
This cell creates a set of plots saved to `figures/`:
- time traces and histograms
- PSD (Welch) linear + log
- spectrogram (i1)
- cross‑correlation (i1 vs i2) and estimated lag
- cross‑spectrum magnitude (coherence proxy)
- group delay raw plot (if present)
Open the saved PNGs to inspect details.

In [ ]:
def save_fig(fig, name):
    p = OUT_DIR / name
    fig.savefig(p, dpi=200, bbox_inches='tight')
    plt.close(fig)
    print('Wrote', p)

# 1) time traces + histogram
fig, axs = plt.subplots(2,1, figsize=(10,6), gridspec_kw={'height_ratios':[2,1]})
axs[0].plot(i1, color='C0', lw=0.8, label='i1')
if i2 is not None:
    axs[0].plot(i2, color='C1', lw=0.8, alpha=0.8, label='i2')
axs[0].legend(); axs[0].set_title('Time domain traces (sample index)')
axs[0].set_ylabel('Amplitude')
axs[1].hist(i1, bins=200, alpha=0.6, label='i1')
if i2 is not None:
    axs[1].hist(i2, bins=200, alpha=0.4, label='i2')
axs[1].legend(); axs[1].set_title('Amplitude histogram')
save_fig(fig, 'explore_time_and_hist.png')

# 2) PSD (Welch)
nperseg = min(4096, max(256, len(i1)//8))
f1, P1 = welch(i1, fs=1.0, nperseg=nperseg)
fig, ax = plt.subplots(1,1, figsize=(10,4))
ax.semilogy(f1, P1, label='i1')
if i2 is not None:
    f2, P2 = welch(i2, fs=1.0, nperseg=nperseg)
    ax.semilogy(f2, P2, label='i2', alpha=0.8)
ax.set_title('Power Spectral Density (Welch)')
ax.set_xlabel('Frequency (sample^-1)')
ax.legend()
save_fig(fig, 'explore_psd.png')

# 3) Spectrogram (i1)
f, t, Sxx = spectrogram(i1, fs=1.0, nperseg=1024, noverlap=768)
fig, ax = plt.subplots(1,1, figsize=(10,3))
pcm = ax.pcolormesh(t, f, 10*np.log10(Sxx+1e-20), shading='gouraud')
ax.set_ylabel('Freq (sample^-1)'); ax.set_xlabel('Time (samples)')
ax.set_title('Spectrogram (i1)')
save_fig(fig, 'explore_spectrogram_i1.png')

# 4) Cross-correlation and lag if i2 exists
if i2 is not None:
    corr = correlate(i1 - np.mean(i1), i2 - np.mean(i2), mode='full')
    lags = np.arange(-len(i1)+1, len(i1))
    lag = lags[np.argmax(corr)]
    fig, ax = plt.subplots(2,1, figsize=(10,6))
    ax[0].plot(lags, corr); ax[0].axvline(lag, color='red', ls='--')
    ax[0].set_title(f'Cross-correlation i1 vs i2 (peak lag = {lag} samples)')
    ax[1].plot(corr[len(corr)//2 - 200: len(corr)//2 + 200])
    ax[1].set_title('Cross-correlation (zoom center)')
    save_fig(fig, 'explore_xcorr.png')

    # 5) cross-spectrum magnitude (proxy coherence)
    C = np.abs(np.fft.rfft(i1) * np.conj(np.fft.rfft(i2)))
    freqs = np.fft.rfftfreq(len(i1), d=1.0)
    fig, ax = plt.subplots(1,1, figsize=(10,3))
    ax.semilogy(freqs, C + 1e-20)
    ax.set_title('Cross-spectrum magnitude (proxy coherence)')
    save_fig(fig, 'explore_cross_spectrum.png')

# 6) group delay raw plot
if group_delay is not None:
    fig, ax = plt.subplots(1,1, figsize=(10,3))
    ax.plot(group_delay); ax.set_title('Group delay (raw index)')
    save_fig(fig, 'explore_group_delay.png')

print('All plots saved to figures/')


## 5) Automated recommendation and next steps
Based on the checks above, the notebook will produce a short recommendation whether the traces are suitable for TD‑GS and what to do next (normalization, resampling, two‑measurement requirement, or NN preprocessing).

In [ ]:
recommendation = {}
status = 'UNKNOWN'

# Basic gating: i1 must be present and finite
if checks['i1_presence'][0] != 'PASS' or checks['i1_finite'][0] != 'PASS':
    recommendation['suitability'] = 'NOT SUITABLE'
    recommendation['reason'] = 'i1 missing or contains non-finite values'
else:
    # length and SNR heuristics
    length_ok = checks['i1_length'][0] == 'PASS'
    snr_ok = checks['i1_snr_proxy'][0] == 'PASS'
    if length_ok and snr_ok:
        # if i2 exists and is not nearly identical, recommend two-measurement GS
        if checks['i2_presence'][0] == 'PASS' and checks['i2_similarity'][0][0] != 'WARN':
            recommendation['suitability'] = 'GOOD'
            recommendation['mode'] = 'single-measurement GS possible; two-measurement GS recommended if dispersions differ'
        elif checks['i2_presence'][0] == 'PASS' and checks['i2_similarity'][0][0] == 'WARN':
            recommendation['suitability'] = 'CAUTION'
            recommendation['mode'] = 'i2 highly similar to i1; may not provide diversity for two-measurement GS. Consider different dispersion or use ML initializer.'
        else:
            recommendation['suitability'] = 'POSSIBLE'
            recommendation['mode'] = 'single-measurement GS OK; two-measurement GS not available (i2 missing)'
    else:
        recommendation['suitability'] = 'MARGINAL'
        recommendation['mode'] = 'Consider increasing sample length or improving SNR; try ML denoiser or averaging.'

# group delay advice
if group_delay is not None:
    recommendation['group_delay'] = 'present; check length vs FFT bins and resample to match frequency axis if needed.'
else:
    recommendation['group_delay'] = 'not present; if you have FBG calibration, include it before two-measurement GS.'

# Save summary JSON
summary_out = {
    'summary_stats': summary,
    'checks': checks,
    'recommendation': recommendation
}
with open(OUT_DIR / 'explore_summary.json', 'w') as f:
    json.dump(summary_out, f, indent=2)

print('Recommendation:')
print(json.dumps(recommendation, indent=2))
print('\nSaved explore_summary.json to figures/')


## 6) Short interpretation guide (how to read the outputs)
- **If `suitability` is `GOOD`**: you can run TD‑GS. Prefer two‑measurement GS if you can confirm the two traces were recorded with different dispersive elements or different settings.
- **If `POSSIBLE` or `MARGINAL`**: try preprocessing: baseline removal, windowing, denoising (simple median or NN denoiser), and re-run the checks. Consider increasing `nfft` or zero‑padding for spectral resolution.
- **If `NOT SUITABLE`**: fix data (remove NaNs, re-acquire, or locate the correct files). TD‑GS requires clean intensity traces.

Open the files in `figures/` and `figures/explore_summary.json` for the numeric details and plots.

## 7) Optional: run a quick TD‑GS test (one cell)
If you want to run a short TD‑GS test from this notebook (quick smoke test), run the next cell. It will run a small number of iterations and plot the reconstructed time trace and error curve. Use this only after you confirm `suitability` is not `NOT SUITABLE`.

In [ ]:
# Quick TD-GS smoke test (uncomment to run)
def temporal_gs_single_quick(measured_time_mag, n_iter=80, pad_factor=1):
    measured_time_mag = np.asarray(measured_time_mag, dtype=np.float64).ravel()
    N = measured_time_mag.size
    nfft = int(1 << (N-1).bit_length()) * pad_factor
    phi = np.random.uniform(-np.pi, np.pi, size=N).astype(np.float64)
    f = (measured_time_mag * np.exp(1j * phi)).astype(np.complex128)
    errs = []
    for _ in range(n_iter):
        F = np.fft.rfft(f, n=nfft)
        f_back = np.fft.irfft(F, n=nfft)[:N].astype(np.complex128)
        err = float(np.mean((np.abs(f_back) - measured_time_mag)**2))
        errs.append(err)
        f = (measured_time_mag * np.exp(1j * np.angle(f_back))).astype(np.complex128)
    return f, np.array(errs)

# To run the smoke test, uncomment the lines below:
# if summary['i1']['shape'] is not None:
#     f_est, errs = temporal_gs_single_quick(np.abs(i1), n_iter=80)
#     fig, ax = plt.subplots(2,1, figsize=(10,6))
#     ax[0].plot(np.arange(i1.size), np.real(i1), label='measured i1')
#     ax[0].plot(np.arange(i1.size), np.real(f_est[:i1.size]), label='GS estimate (real)', alpha=0.8)
#     ax[0].legend(); ax[0].set_title('Time domain: measured vs GS estimate')
#     ax[1].plot(errs); ax[1].set_title('GS squared-error vs iteration')
#     plt.tight_layout(); plt.show()

print('Smoke test cell ready. Uncomment and run if you want a quick TD-GS trial.')


### End of notebook

If you want, I can now:
- Export this notebook as a `.ipynb` file for download, or
- Convert the key cells into a single robust script (`tools/explore_dryad.py`) and provide the exact PowerShell command to run it, or
- Run the quick TD‑GS smoke test and show the plots here (if you confirm).